# `trl` smoke test

`trl` hard-depends on `transformers` and `datasets`; every cell here exercises one or both, so a broken
install of either fails this notebook too.

In [ ]:
# Example 1 — the SFT, DPO and Reward config classes build and keep their settings (CPU-only).
import os, tempfile
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
from trl import SFTConfig, DPOConfig, RewardConfig
with tempfile.TemporaryDirectory() as out:
    sft = SFTConfig(output_dir=out, max_length=32, packing=True, per_device_train_batch_size=2, use_cpu=True)
    dpo = DPOConfig(output_dir=out, beta=0.1, max_length=32, use_cpu=True)
    rew = RewardConfig(output_dir=out, max_length=32, per_device_train_batch_size=2, use_cpu=True)
assert sft.max_length == 32 and sft.packing is True
assert abs(dpo.beta - 0.1) < 1e-9
assert rew.per_device_train_batch_size == 2
print("configs OK:", sft.__class__.__name__, dpo.__class__.__name__, rew.__class__.__name__)

In [ ]:
# Example 2 — SFTTrainer runtime init + one real training step on a tiny Llama.
# Everything (tokenizer, model, dataset) is built in-cell; no weights are downloaded.
import tempfile, torch
from tokenizers import Tokenizer, models, pre_tokenizers
from transformers import PreTrainedTokenizerFast, LlamaConfig, LlamaForCausalLM
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
vocab = {"[PAD]": 0, "[EOS]": 1, "[UNK]": 2}
for i, w in enumerate("the quick brown fox jumps over lazy dog hello world".split()): vocab[w] = 3 + i
tk = Tokenizer(models.WordLevel(vocab=vocab, unk_token="[UNK]")); tk.pre_tokenizer = pre_tokenizers.Whitespace()
tok = PreTrainedTokenizerFast(tokenizer_object=tk, pad_token="[PAD]", eos_token="[EOS]", unk_token="[UNK]")
cfg = LlamaConfig(vocab_size=len(vocab), hidden_size=16, intermediate_size=32, num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=2, max_position_embeddings=64)
model = LlamaForCausalLM(cfg)
ds = Dataset.from_dict({"text": ["the quick brown fox", "hello world dog"] * 4})
with tempfile.TemporaryDirectory() as out:
    args = SFTConfig(output_dir=out, max_steps=1, per_device_train_batch_size=2, learning_rate=1e-3, report_to=[], logging_strategy="no", save_strategy="no", max_length=16, use_cpu=True)
    result = SFTTrainer(model=model, args=args, train_dataset=ds, processing_class=tok).train()
assert result.global_step == 1
print(f"SFT 1-step OK — loss={result.training_loss:.3f}")

assert torch.cuda.is_available(), (
    f"requires-gpu cell ran but torch.cuda.is_available() is False — torch {torch.__version__}, "
    f"built against CUDA {torch.version.cuda}, device_count={torch.cuda.device_count()}"
)

logits = model.to("cuda")(input_ids=torch.randint(0, len(vocab), (1, 4), device="cuda")).logits
print("GPU forward:", tuple(logits.shape))

In [ ]:
# Example 3 — the pure-Python reward functions (trl.rewards). CPU-only, no model needed.
from trl.rewards import think_format_reward, get_repetition_penalty_reward, get_soft_overlong_punishment
# Conversational-format reward: a <think>...</think> block present -> 1.0, absent -> 0.0.
conv = [[{"role": "assistant", "content": "<think>reason</think>answer"}],
        [{"role": "assistant", "content": "answer with no thinking block"}]]
assert think_format_reward(completions=conv) == [1.0, 0.0]
# Repetition penalty over token-id n-grams: repetitive -> negative, diverse -> 0.0.
rep = get_repetition_penalty_reward(ngram_size=2, max_penalty=-1.0)
rep_vals = rep(completion_ids=[[1, 1, 1, 1, 1, 1], [1, 2, 3, 4, 5, 6]])
assert rep_vals[0] < 0.0 and rep_vals[1] == 0.0
# Soft overlong punishment: within the length budget -> 0.0, overlong -> penalized.
sop = get_soft_overlong_punishment(max_completion_len=10, soft_punish_cache=3)
sop_vals = sop(completion_ids=[list(range(5)), list(range(12))])
assert sop_vals[0] == 0.0 and sop_vals[1] == -1.0
print("trl.rewards OK:", think_format_reward(completions=conv), rep_vals, sop_vals)

In [ ]:
# Example 4 — chat-template data utilities (trl.data_utils): detection, rendering, round-trip.
from tokenizers import Tokenizer, models, pre_tokenizers
from transformers import PreTrainedTokenizerFast
from trl.data_utils import is_conversational, apply_chat_template, maybe_apply_chat_template
tk = Tokenizer(models.WordLevel(vocab={"[PAD]": 0, "[UNK]": 1, "hi": 2, "there": 3}, unk_token="[UNK]"))
tk.pre_tokenizer = pre_tokenizers.Whitespace()
template = "{% for m in messages %}{{ m.role }}: {{ m.content }}\n{% endfor %}"
tok = PreTrainedTokenizerFast(tokenizer_object=tk, pad_token="[PAD]", unk_token="[UNK]", eos_token="[UNK]", chat_template=template)
conv = {"messages": [{"role": "user", "content": "hi there"}]}
plain = {"prompt": "hi there"}
assert is_conversational(conv) is True and is_conversational(plain) is False
rendered = apply_chat_template(conv, tok)
assert rendered["text"] == "user: hi there\n"
# maybe_apply_chat_template is a no-op on an already-flat (non-conversational) example.
assert maybe_apply_chat_template(plain, tok) == plain
print("data_utils OK — rendered:", repr(rendered["text"]))

In [ ]:
# Example 5 — a second trainer family: RewardTrainer on a preference (chosen/rejected) pair.
# Exercises trl's sequence-classification reward-modeling path, distinct from SFT.
import tempfile
from tokenizers import Tokenizer, models, pre_tokenizers
from transformers import PreTrainedTokenizerFast, LlamaConfig, LlamaForSequenceClassification
from datasets import Dataset
from trl import RewardConfig, RewardTrainer
vocab = {"[PAD]": 0, "[UNK]": 1}
for i, w in enumerate("good bad hi there the a an dog".split()): vocab[w] = 2 + i
tk = Tokenizer(models.WordLevel(vocab=vocab, unk_token="[UNK]")); tk.pre_tokenizer = pre_tokenizers.Whitespace()
tok = PreTrainedTokenizerFast(tokenizer_object=tk, pad_token="[PAD]", unk_token="[UNK]", eos_token="[UNK]")
cfg = LlamaConfig(vocab_size=len(vocab), hidden_size=16, intermediate_size=32, num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=2, max_position_embeddings=64, num_labels=1, pad_token_id=0)
model = LlamaForSequenceClassification(cfg)
ds = Dataset.from_dict({"chosen": ["hi there good"] * 4, "rejected": ["hi there bad"] * 4})
with tempfile.TemporaryDirectory() as out:
    args = RewardConfig(output_dir=out, max_steps=1, per_device_train_batch_size=2, report_to=[], logging_strategy="no", save_strategy="no", use_cpu=True, max_length=16)
    result = RewardTrainer(model=model, args=args, train_dataset=ds, processing_class=tok).train()
assert result.global_step == 1
print(f"RewardTrainer 1-step OK — loss={result.training_loss:.3f}")